<div align="center">
<h1 style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 26</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">IPsec və VPN</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">Təhlükəsiz Şəbəkə Əlaqələri</h3>
<br>
<div style="background-color: #2a5298; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024 | Yenilənmiş versiya</em></p>
</div>

## 📑 Mündəricat

1. Məqsədlər və icra müqaviləsi
2. IPsec arxitekturası: SA, AH, ESP və rejimlər
3. NAT-T və IKEv2 exchanges
4. VPN və cryptokey routing modeli
5. WireGuard sərhədləri və test açarları
6. Ədalətli primitive microbenchmark
7. Konfiqurasiya review-u və özünü-yoxlama
8. Ev tapşırığı və mənbələr


## 🎯 Məşğələnin məqsədləri

- SA-nı birtərəfli state kimi izah etmək;
- AH və ESP wire sahələrini RFC 4302/4303 üzrə ayırmaq;
- ESP trailer-in şifrləmədən əvvəl payload-a əlavə olunduğunu göstərmək;
- IKEv2 `IKE_SA_INIT`, `IKE_AUTH`, `CREATE_CHILD_SA` və `INFORMATIONAL` exchanges-i ayırmaq;
- NAT detection hash-ini SPI, IP və port üzrə hesablamaq;
- WireGuard public key/AllowedIPs əlaqəsini protocol implementation-dan ayırmaq;
- yalnız eyni security primitive sinfində olan AEAD-ləri microbenchmark etmək.


## 📚 İcra mühiti

Notebook standard library və `cryptography` istifadə edir. Paket quraşdırmır, şəbəkəyə çıxmır, administrator komandası icra etmir, `cwd` dəyişmir və private key çıxarmır. AH/ESP və routing kodu **məhdud struktur modelidir**, kernel IPsec və ya WireGuard implementation deyil.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec

if find_spec("cryptography") is None:
    raise RuntimeError("Bu notebook üçün 'cryptography' tələb olunur")
try:
    CRYPTOGRAPHY_VERSION = version("cryptography")
except PackageNotFoundError as exc:
    raise RuntimeError("cryptography paylanması tapılmadı") from exc
print(f"cryptography {CRYPTOGRAPHY_VERSION} hazırdır")


## 🔧 Hazırlıq

Hər code cell assertion-larla öz müqaviləsini yoxlayır. Kriptoqrafik təsadüfilik nəticələri çıxışa verilmir. Benchmark yalnız bu maşında istifadə olunan backend üçün lokal müşahidədir; VPN throughput, latency və ya IPsec/WireGuard üstünlüyü kimi şərh edilmir.


### 3.1 API yoxlaması

Məcburi API-lər əvvəlcədən import olunur. Optional skip və broad exception yoxdur; gözlənilən authentication xətası ayrıca `InvalidTag` ilə tutulur.


In [ ]:
import base64
import hashlib
import ipaddress
import secrets
import struct
import sys
import time
from statistics import median

import cryptography
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, ChaCha20Poly1305

assert AESGCM and ChaCha20Poly1305 and X25519PrivateKey
print(f"Python {sys.version_info.major}.{sys.version_info.minor}; cryptography {cryptography.__version__}")


### 3.2 Laboratoriya sərhədi

Real IPsec ESP/AH emalı OS kernelində və ya audit edilmiş implementation-da aparılır. Bu notebook yalnız seçilmiş header/trailer invariantlarını, AEAD round-trip-i, NAT hash input-u və routing-policy məntiqini göstərir.


In [ ]:
LAB_CONTRACT = {
    "external_network": False,
    "admin_commands": False,
    "persistent_secrets": False,
    "kernel_vpn_claim": False,
    "deterministic_assertions": True,
}
assert not any(LAB_CONTRACT[key] for key in ("external_network", "admin_commands", "persistent_secrets", "kernel_vpn_claim"))
assert LAB_CONTRACT["deterministic_assertions"]
print("Lokal model müqaviləsi təsdiqləndi")


## 📖 IPsec nədir?

IPsec IP qatında traffic selectors və Security Associations əsasında paketləri qoruyan arxitekturadır. Məxfilik, bütövlük, data-origin authentication və anti-replay xassələri seçilmiş ESP/AH profilinə görə fərqlənir. “VPN” daha geniş deployment anlayışıdır; hər VPN IPsec deyil.


### Trust və deployment sərhədi

IPsec endpoint və ya security gateway-lər arasında işləyə bilər. Təhlükəsizlik yalnız alqoritmdən deyil, peer authentication, traffic selector authorization, SA lifecycle, anti-replay window, routing və host firewall siyasətindən asılıdır.


## 🏛️ IPsec arxitekturası

Əsas anlayışlar: SPD (hansı traffic qorunur/bypass/drop), SAD (aktiv SA state), IKEv2 (IKE SA və Child SA yaradılması), ESP və daha az istifadə olunan AH. Inbound və outbound emal ayrıca siyasət və state tələb edir.


### 4.1 Əsas komponentlər

| Komponent | Rol |
|---|---|
| SPD | Traffic selector üzrə protect/bypass/discard qərarı |
| SAD | SPI, peer, alqoritm, key, sequence və anti-replay state |
| IKEv2 | Peer authentication və SA negotiation/rekey/delete |
| ESP | Məxfilik və/və ya bütövlük/authentication profili |
| AH | Məxfilik vermədən integrity/data-origin authentication; NAT ilə problemli |


### 4.2 Security Association

SA birtərəflidir. Praktik ikiistiqamətli qoruma ən azı iki Child SA tələb edir. SPI təkbaşına qlobal identity deyil; destination və IPsec protocol ilə birlikdə inbound SA lookup-a kömək edir. Sequence number anti-replay state ilə istifadə olunur və key/SA reuse sərhədlərinə tabedir.


### 4.3 AH — IP protocol 51

AH header: Next Header, Payload Len, Reserved, SPI, Sequence Number və dəyişən ICV. `Payload Len = AH uzunluğu / 4 - 2`. ICV hesabında ICV sahəsi sıfırlanır, immutable/predictable IP sahələri və payload əhatə olunur; dəyişə bilən IP sahələrinin canonicalization qaydaları RFC 4302-də ayrıca verilir.


In [ ]:
def build_ah_header(next_header, spi, sequence, icv_length=16):
    if icv_length % 4 != 0 or icv_length < 12:
        raise ValueError("Demo ICV uzunluğu 4-bayt aligned və ən azı 12 olmalıdır")
    total_length = 12 + icv_length
    payload_len = total_length // 4 - 2
    zero_icv = bytes(icv_length)
    return struct.pack("!BBHII", next_header, payload_len, 0, spi, sequence) + zero_icv


def parse_ah_header(header):
    if len(header) < 24 or len(header) % 4:
        raise ValueError("AH header uzunluğu yanlışdır")
    next_header, payload_len, reserved, spi, sequence = struct.unpack("!BBHII", header[:12])
    expected_length = (payload_len + 2) * 4
    if expected_length != len(header) or reserved != 0:
        raise ValueError("AH Payload Len/Reserved yanlışdır")
    return {"next_header": next_header, "payload_len": payload_len, "spi": spi, "sequence": sequence, "icv_length": len(header) - 12}


ah_header = build_ah_header(6, 0x10203040, 7)
ah_result = parse_ah_header(ah_header)
assert ah_result == {"next_header": 6, "payload_len": 5, "spi": 0x10203040, "sequence": 7, "icv_length": 16}
print(ah_result)


### 4.4 ESP — IP protocol 50

ESP top-level sahələri: SPI, Sequence Number, Payload Data, Padding, Pad Length, Next Header və optional/integrated authentication data. Padding, Pad Length və Next Header **şifrlənən plaintext-in içindədir**. AEAD profilində SPI/sequence AAD-a daxil ola bilər; nonce quruluşu konkret transform RFC-si ilə müəyyən edilir.


In [ ]:
def esp_pad(payload, next_header, alignment=4):
    pad_length = (-(len(payload) + 2)) % alignment
    padding_bytes = bytes(range(1, pad_length + 1))
    return payload + padding_bytes + bytes([pad_length, next_header])


def esp_aes_gcm_protect(key, salt, explicit_iv, spi, sequence, payload, next_header):
    if len(salt) != 4 or len(explicit_iv) != 8:
        raise ValueError("Bu RFC 4106 modeli 4-byte salt və 8-byte explicit IV tələb edir")
    header = struct.pack("!II", spi, sequence)
    plaintext = esp_pad(payload, next_header)
    ciphertext_and_tag = AESGCM(key).encrypt(salt + explicit_iv, plaintext, header)
    return header + explicit_iv + ciphertext_and_tag


def esp_aes_gcm_open(key, salt, packet):
    header, explicit_iv, ciphertext_and_tag = packet[:8], packet[8:16], packet[16:]
    plaintext = AESGCM(key).decrypt(salt + explicit_iv, ciphertext_and_tag, header)
    pad_length, next_header = plaintext[-2], plaintext[-1]
    padding_bytes = plaintext[-2 - pad_length:-2]
    if padding_bytes != bytes(range(1, pad_length + 1)):
        raise ValueError("ESP padding pattern yanlışdır")
    return plaintext[:-2 - pad_length], next_header


esp_key, esp_salt, esp_iv = AESGCM.generate_key(bit_length=128), secrets.token_bytes(4), secrets.token_bytes(8)
esp_packet = esp_aes_gcm_protect(esp_key, esp_salt, esp_iv, 0xA0B0C0D0, 1, b"inner-packet", 4)
esp_plaintext, esp_next_header = esp_aes_gcm_open(esp_key, esp_salt, esp_packet)
assert esp_plaintext == b"inner-packet" and esp_next_header == 4
tampered_esp_rejected = False
try:
    esp_aes_gcm_open(esp_key, esp_salt, esp_packet[:-1] + bytes([esp_packet[-1] ^ 1]))
except InvalidTag:
    tampered_esp_rejected = True
assert tampered_esp_rejected
print({"packet_bytes": len(esp_packet), "round_trip": True, "tamper_rejected": True})


### 4.5 Transport və tunnel rejimi

- **Transport:** original IP header qalır; ESP adətən üst-qat payload-ını qoruyur.
- **Tunnel:** bütöv inner IP packet yeni outer IP packet-in ESP payload-ı olur.

Outer header routing üçün görünür. Tunnel mode “bütün metadata gizlidir” demək deyil; packet size, timing və outer endpoints görünə bilər.


In [ ]:
def ipsec_mode_model(original_ip_header, upper_layer_payload, new_outer_header):
    inner_packet = original_ip_header + upper_layer_payload
    return {
        "transport": {"visible_outer": original_ip_header, "esp_plaintext_model": upper_layer_payload},
        "tunnel": {"visible_outer": new_outer_header, "esp_plaintext_model": inner_packet},
    }


mode_result = ipsec_mode_model(b"IPv4-A-to-B", b"TCP+DATA", b"IPv4-GW-to-GW")
assert mode_result["transport"]["esp_plaintext_model"] == b"TCP+DATA"
assert mode_result["tunnel"]["esp_plaintext_model"] == b"IPv4-A-to-BTCP+DATA"
print({mode: list(value) for mode, value in mode_result.items()})


## 🌐 NAT Traversal

NAT IP ünvanı/portu dəyişdirdiyi üçün AH ICV-si və native ESP demultiplexing problem yarada bilər. IKEv2 NAT detection notify-ləri SPIi || SPIr || IP || port üzərində SHA-1 digest daşıyır. Mismatch NAT ehtimalını göstərir; IKE və ESP UDP/4500 encapsulation-a keçir. Bu hash məxfi ünvanı güclü gizlətmir.


### ✍️ Çalışma 1 — 1 bal

AH `Payload Len` üçün 12, 16 və 32-byte ICV nümunələri, ESP üçün isə 0–7 bayt payload uzunluqları yaradın.

**Qəbul meyarı:** AH uzunluğu formula ilə uyğun gəlir; ESP padding/trailer encrypt-dən əvvəldir; hər ESP round-trip və tamper rejection keçir.


## 🔑 IKEv2

IKEv2 IKE SA qurur və onun qoruması altında ESP/AH Child SA-ları yaradır. IKE SA və Child SA eyni state deyil. Peer authentication olmadan yalnız `IKE_SA_INIT` nəticəsinə tam identity etimadı verilməməlidir.


### 5.1 Exchanges

1. `IKE_SA_INIT` request/response: algorithms, nonces, DH və NAT detection.
2. `IKE_AUTH` request/response: əvvəlki exchange authentication-u, identities/certificates və ilk Child SA.
3. `CREATE_CHILD_SA`: yeni/rekey Child SA və ya IKE SA rekey.
4. `INFORMATIONAL`: delete, error və digər idarəetmə mesajları.

İlkin minimal axın adətən dörd mesajdır, lakin cookies, EAP və başqa variasiyalar mesaj sayını artıra bilər.


### 5.2 IKEv1 və IKEv2

IKEv2 exchange/state modelini sadələşdirir, MOBIKE/EAP və etibarlı retransmission semantikası üçün müasir əsasdır. “IKEv2 həmişə dörd mesajdır” doğru deyil. IKEv1 terminlərini (`Main/Aggressive/Quick Mode`) IKEv2 exchange adları ilə qarışdırmayın.


In [ ]:
def ikev2_nat_detection_hash(spi_i, spi_r, ip_address, port):
    spi_bytes = spi_i.to_bytes(8, "big") + spi_r.to_bytes(8, "big")
    ip_bytes = ipaddress.ip_address(ip_address).packed
    port_bytes = port.to_bytes(2, "big")
    return hashlib.sha1(spi_bytes + ip_bytes + port_bytes).digest()


spi_i, spi_r = 0x0102030405060708, 0x1112131415161718
sent_nat_hash = ikev2_nat_detection_hash(spi_i, spi_r, "10.0.0.10", 500)
observed_same = ikev2_nat_detection_hash(spi_i, spi_r, "10.0.0.10", 500)
observed_nat = ikev2_nat_detection_hash(spi_i, spi_r, "198.51.100.7", 4500)
ikev2_result = {
    "same_path_matches": sent_nat_hash == observed_same,
    "translated_path_mismatch": sent_nat_hash != observed_nat,
    "initial_exchanges": ["IKE_SA_INIT", "IKE_AUTH"],
    "subsequent": ["CREATE_CHILD_SA", "INFORMATIONAL"],
}
assert all((ikev2_result["same_path_matches"], ikev2_result["translated_path_mismatch"]))
print(ikev2_result)


### ✍️ Çalışma 2 — 1 bal

Eyni SPI-lərlə source port, destination IP və SPIr-i ayrı-ayrılıqda dəyişin.

**Qəbul meyarı:** hər dəyişiklik digest-i dəyişir; input byte order göstərilir; hash authentication və ya confidentiality kimi təqdim edilmir.


## 🌐 VPN anlayışı

VPN untrusted və ya paylaşılan şəbəkə üzərində seçilmiş traffic üçün virtual connectivity və security policy yaradır. Məhsullar remote-access, site-to-site, overlay və application tunnel modellərindən istifadə edə bilər; təhlükəsizlik xassəsi konkret protocol və authentication siyasətindən gəlir.


### 6.1 VPN nəyi avtomatik vermir?

VPN endpoint cihazını malware-dən, DNS/route səhvindən, zəif identity siyasətindən və traffic correlation-dan avtomatik qorumur. “Bütün trafik şifrələnir” yalnız selector/route və fail behavior bunu təmin edirsə doğrudur.


### 6.2 Növlər və trust boundary

| Model | Tipik sərhəd |
|---|---|
| Remote access | Client ↔ gateway; endpoint posture ayrıca |
| Site-to-site | Gateway ↔ gateway; inner subnet authorization ayrıca |
| Host-to-host | İki host; local service policy ayrıca |
| Overlay/mesh | Peer identity + route distribution; control plane ayrıca |


In [ ]:
def longest_prefix_peer(destination, peer_routes):
    address = ipaddress.ip_address(destination)
    matches = []
    for peer, networks in peer_routes.items():
        for network_text in networks:
            network = ipaddress.ip_network(network_text)
            if address in network:
                matches.append((network.prefixlen, peer, str(network)))
    if not matches:
        return None
    _, peer, network = max(matches)
    return {"peer": peer, "matched_network": network}


peer_routes = {"peer-a": ["10.0.0.0/8"], "peer-b": ["10.20.0.0/16"], "peer-c": ["192.0.2.0/24"]}
routing_result = longest_prefix_peer("10.20.30.40", peer_routes)
assert routing_result == {"peer": "peer-b", "matched_network": "10.20.0.0/16"}
assert longest_prefix_peer("203.0.113.1", peer_routes) is None
print(routing_result)


## ⚠️ VPN hücum səthi

Əsas risklər: credential/key oğurlanması, selector/AllowedIPs səhvi, route və DNS leak, downgrade/legacy profile, replay/state qüsuru, endpoint compromise, management-plane exposure və traffic analysis. Müdafiə protocol-dan əlavə inventory, patching, least privilege, MFA/peer-key lifecycle, monitoring və fail-closed routing tələb edə bilər.


### ✍️ Çalışma 3 — 1 bal

Bir remote-access və bir site-to-site deployment üçün asset, trust boundary, selector, failure mode və monitorinq sübutu cədvəli hazırlayın.

**Qəbul meyarı:** ən azı route leak, stolen credential və endpoint compromise ayrılır; VPN-in nəyi həll etmədiyi göstərilir.


## ⚡ WireGuard

WireGuard Linux kernel implementation və digər portlarla yayılan VPN protokoludur; “WireGuard RFC (2020)” adlı IETF standartı yoxdur. Əsas primary mənbə rəsmi protocol whitepaper və implementation sənədləridir.


### 7.1 Cryptokey Routing

Peer public key `AllowedIPs` siyahısı ilə əlaqələndirilir. Göndərmədə bu siyahı route seçiminə, qəbulda isə source-address authorization-a təsir edir. Overlap və route policy diqqətlə yoxlanmalıdır; public key insan identity-sini öz-özünə sübut etmir.


### 7.2 Kriptoqrafik sərhəd

WireGuard Noise_IK əsaslı handshake və Curve25519, ChaCha20-Poly1305, BLAKE2s, SipHash24, HKDF kimi primitive-lərdən istifadə edir. Bu adlar ayrıca “WireGuard implementation” deyil. Private key 32 raw baytdır və konfiqurasiyada base64 kimi göstərilir; key lifecycle ayrıca idarə olunur.


In [ ]:
def wireguard_test_keypair():
    private_key = X25519PrivateKey.generate()
    private_raw = private_key.private_bytes(
        serialization.Encoding.Raw,
        serialization.PrivateFormat.Raw,
        serialization.NoEncryption(),
    )
    public_raw = private_key.public_key().public_bytes(
        serialization.Encoding.Raw,
        serialization.PublicFormat.Raw,
    )
    private_b64 = base64.b64encode(private_raw).decode("ascii")
    public_b64 = base64.b64encode(public_raw).decode("ascii")
    assert len(base64.b64decode(private_b64, validate=True)) == 32
    assert len(base64.b64decode(public_b64, validate=True)) == 32
    return private_key, public_b64


wg_private_key, wg_public_key_b64 = wireguard_test_keypair()
wireguard_config_template = f"""[Interface]
PrivateKey = <INJECT_FROM_SECRET_STORE>
Address = 10.7.0.1/24

[Peer]
PublicKey = {wg_public_key_b64}
AllowedIPs = 10.8.0.0/24
Endpoint = vpn.example.test:51820
"""
assert len(wg_public_key_b64) == 44 and "<INJECT_FROM_SECRET_STORE>" in wireguard_config_template
print({"public_key_base64_chars": len(wg_public_key_b64), "private_key_printed": False})


### 7.3 IPsec və WireGuard müqayisəsi

IPsec/IKEv2 geniş algorithm/profile və enterprise integration seçimlərinə, WireGuard isə daha kiçik sabit protocol səthinə malikdir. Throughput və latency hardware acceleration, kernel/userspace path, MTU, packet size, CPU, NIC offload və deployment-dan asılıdır. Primitive microbenchmark ilə protocol qalibi elan edilmir.


### ✍️ Çalışma 4 — 1 bal

Üç peer üçün public key və non-overlapping `AllowedIPs` inventarı qurun.

**Qəbul meyarı:** hər public key 32-byte base64 kimi yoxlanır; private key output yoxdur; overlap və missing route mənfi testləri var.


## 📊 Ədalətli AEAD microbenchmark

AES-GCM və ChaCha20-Poly1305 eyni payload, AAD, round-trip və təkrar sayı ilə ölçülür. Bu yalnız Python/OpenSSL backend müşahidəsidir. AES hardware acceleration və platforma fərqi nəticəni dəyişə bilər; IPsec və WireGuard performansı buradan çıxarılmır.


In [ ]:
def benchmark_aead(payload_size=16_384, rounds=31):
    payload = bytes([0xA5]) * payload_size
    aad = b"AS26-comparable-AEAD"
    algorithms = {
        "AES-256-GCM": AESGCM(AESGCM.generate_key(bit_length=256)),
        "ChaCha20-Poly1305": ChaCha20Poly1305(ChaCha20Poly1305.generate_key()),
    }
    results = {}
    for name, algorithm in algorithms.items():
        samples = []
        for counter in range(rounds + 5):
            nonce = counter.to_bytes(12, "big")
            start = time.perf_counter_ns()
            ciphertext = algorithm.encrypt(nonce, payload, aad)
            recovered = algorithm.decrypt(nonce, ciphertext, aad)
            elapsed = time.perf_counter_ns() - start
            assert recovered == payload and len(ciphertext) == len(payload) + 16
            if counter >= 5:
                samples.append(elapsed)
        results[name] = {"median_round_trip_ns": int(median(samples)), "samples": len(samples), "payload_bytes": payload_size}
    return results


benchmark_result = benchmark_aead()
assert all(value["median_round_trip_ns"] > 0 and value["samples"] == 31 for value in benchmark_result.values())
print(benchmark_result)


### ✍️ Çalışma 5 — 1 bal

64, 1024 və 16384-byte payload-ları ən azı 31 sample ilə ölçün.

**Qəbul meyarı:** warm-up və median var; eyni guarantees müqayisə olunur; nəticə platforma/backend ilə qeyd edilir; protocol throughput və “N dəfə sürətli” nəticəsi çıxarılmır.


## 🛠️ Konfiqurasiya review-u

Notebook root/admin əmri icra etmir. Aşağıdakı obyektlər yalnız review fixture-dir. Real StrongSwan/swanctl və WireGuard syntax versiya və distribution sənədləri ilə yoxlanmalı, secret store, firewall, routes, MTU və rollback ayrıca hazırlanmalıdır.


In [ ]:
def strongswan_policy_fixture():
    return {
        "ike_version": 2,
        "local_id": "gw-a.example.test",
        "remote_id": "gw-b.example.test",
        "local_ts": ["10.10.0.0/16"],
        "remote_ts": ["10.20.0.0/16"],
        "authentication": "certificate",
        "ike_proposal_example": "aes256gcm16-prfsha384-ecp384",
        "esp_proposal_example": "aes256gcm16-ecp384",
        "secrets_in_repository": False,
    }


strongswan_policy = strongswan_policy_fixture()
assert strongswan_policy["ike_version"] == 2 and not strongswan_policy["secrets_in_repository"]
print(strongswan_policy)


In [ ]:
def wireguard_policy_review(config_text):
    required = ("[Interface]", "[Peer]", "PrivateKey", "PublicKey", "AllowedIPs")
    return {
        "required_fields_present": all(token in config_text for token in required),
        "private_key_is_placeholder": "<INJECT_FROM_SECRET_STORE>" in config_text,
        "public_key_decodes_to_32_bytes": len(base64.b64decode(wg_public_key_b64, validate=True)) == 32,
        "no_commands_executed": True,
    }


wireguard_policy = wireguard_policy_review(wireguard_config_template)
assert all(wireguard_policy.values())
print(wireguard_policy)


### ✍️ Çalışma 6 — 1 bal

Bir IPsec və bir WireGuard policy fixture üçün peer identity, selectors/routes, key source, rekey/replacement, firewall və rollback checklist-i yazın.

**Qəbul meyarı:** secret repozitoriyada deyil; konkret deployment syntax “universal” adlandırılmır; dəyişikliklər icra edilmir.


## 🖥️ İnteqrasiya edilmiş özünü-yoxlama

Self-check yalnız notebook-un struktur, AEAD, NAT hash, routing, key format və benchmark invariantlarını əhatə edir. Kernel SA negotiation, real packet capture və VPN throughput yoxlanmır.


In [ ]:
def run_ipsec_vpn_self_check():
    checks = {
        "ah_layout": ah_result["payload_len"] == 5,
        "esp_round_trip": esp_plaintext == b"inner-packet" and esp_next_header == 4,
        "esp_tamper_rejection": tampered_esp_rejected,
        "mode_boundary": mode_result["transport"] != mode_result["tunnel"],
        "nat_detection": ikev2_result["translated_path_mismatch"],
        "longest_prefix": routing_result["peer"] == "peer-b",
        "wireguard_key_format": len(base64.b64decode(wg_public_key_b64, validate=True)) == 32,
        "comparable_benchmark": set(benchmark_result) == {"AES-256-GCM", "ChaCha20-Poly1305"},
        "strongswan_policy": not strongswan_policy["secrets_in_repository"],
        "wireguard_policy": all(wireguard_policy.values()),
    }
    assert all(checks.values())
    return checks


ipsec_vpn_checks = run_ipsec_vpn_self_check()
print({"passed": sum(ipsec_vpn_checks.values()), "total": len(ipsec_vpn_checks)})


## 🏠 Ev tapşırığı — 10 bal

RFC-lərə əsaslanan site-to-site design review hazırlayın.

| Meyar | Bal |
|---|---:|
| SPD/SAD və ikiistiqamətli Child SA modeli | 2 |
| ESP profile, sequence/anti-replay və NAT-T | 2 |
| IKEv2 identity, exchange və rekey modeli | 2 |
| Traffic selectors, routes və fail behavior | 1 |
| Üç mənfi test: tamper, wrong peer, route overlap | 2 |
| Secret lifecycle və rollback planı | 1 |

Kod yalnız lokal modeldir; admin əmri və real interface dəyişiklikləri qadağandır.


## 📌 Yekun

- SA birtərəflidir; iki istiqamət ayrıca state tələb edir.
- ESP padding/trailer encrypt-dən əvvəl plaintext-ə daxildir.
- AH məxfilik vermir və NAT dəyişiklikləri ilə uyğunlaşmır.
- IKEv2 NAT detection SPI, IP və port hash-lərini müqayisə edir.
- İlkin IKEv2 axını adətən 4 mesajdır, həmişə deyil.
- WireGuard IETF RFC-si yoxdur; rəsmi whitepaper primary mənbədir.
- Primitive microbenchmark VPN performans müqayisəsi deyil.


## 📚 Əlavə resurslar

- RFC 4301 — Security Architecture for IP
- RFC 4302 — IP Authentication Header
- RFC 4303 — IP Encapsulating Security Payload
- RFC 4106 — AES-GCM ESP profile
- RFC 7296 — IKEv2
- RFC 8221 — Cryptographic Algorithm Implementation Requirements for ESP/AH
- RFC 8247 — Algorithm Implementation Requirements for IKEv2
- RFC 9333 — Minimal ESP
- WireGuard official protocol whitepaper və documentation

Cari production profile üçün RFC updates, vendor documentation və OS/kernel versiyası ayrıca yoxlanmalıdır.
